# Import necessary libraries

In [1]:
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from joblib import dump, load

# Load dataset

In [2]:
df = pd.read_csv('../data/training_data.csv')
df.head()

,Diabetes,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,HeartDiseaseorAttack,PhysActivity,HvyAlcoholConsump,AnyHealthcare,NoDocbcCost,GenHlth,MentHlth,PhysHlth,DiffWalk,Sex,Age,Education,Income
0,0.0,0.0,1.0,1.0,15.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,5.0,10.0,20.0,0.0,0.0,11.0,4.0,5.0
1,1.0,1.0,0.0,1.0,28.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,2.0,0.0,0.0,0.0,0.0,11.0,4.0,3.0
2,1.0,1.0,1.0,1.0,33.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,2.0,10.0,0.0,0.0,0.0,9.0,4.0,7.0
3,1.0,0.0,1.0,1.0,29.0,0.0,1.0,1.0,1.0,0.0,1.0,0.0,5.0,0.0,30.0,1.0,1.0,12.0,3.0,4.0
4,0.0,0.0,0.0,1.0,24.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,3.0,0.0,0.0,1.0,1.0,13.0,5.0,6.0


# Define training-related stuffs

In [3]:
LABEL_COL = 'Diabetes'
CATEGORICAL_COLS = ['GenHlth', 'Age', 'Education', 'Income']
NUMERICAL_COLS = ['BMI', 'MentHlth', 'PhysHlth']

TEST_SIZE = 0.2
LR = 1e-3
L2_REG = 1e-4
MAX_DEPTH = 20
NUM_EPOCHS = 1000
PATIENCE = 20
MIN_CHILD_WEIGHT = 5
COSAMPLE_RATE = 0.8
DATA_RATE = 0.8

# Data preprocessing

In [4]:
# Separate features and labels
X = df.drop(columns=[LABEL_COL])
y = df[LABEL_COL].values

# Store one encoder per column
encoders = {}
for col in CATEGORICAL_COLS:
    encoder = LabelEncoder()
    X[col] = encoder.fit_transform(X[col])
    encoders[col] = encoder  # Save each encoder with its column name
# Save all encoders as a dictionary
dump(encoders, '../models/label_encoders.bin')

# How to load encoder
# # Load all encoders
# encoders = load('../models/label_encoders.bin')
# # Transform new data
# for col in CATEGORICAL_COLS:
#     X_new[col] = encoders[col].transform(X_new[col])
    
# Normalize numerical features
scaler = StandardScaler()
X[NUMERICAL_COLS] = scaler.fit_transform(X[NUMERICAL_COLS])
# Save the scaler for future use
dump(scaler, '../models/scaler.bin')

# How to load scaler
# # Load scaler
# scaler = load('../models/scaler.bin')
# # Transform new data
# X_new[NUMERICAL_COLS] = scaler.transform(X_new[NUMERICAL_COLS])

# Train/validation/test split
# First split: separate test set
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, 
    test_size=TEST_SIZE * 2, 
    random_state=42, 
    stratify=y
)

# Second split: separate validation set from remaining data
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, 
    test_size=0.5, 
    random_state=42, 
    stratify=y_temp
)

print(f"Training set size: {len(X_train)}")
print(f"Validation set size: {len(X_val)}")
print(f"Test set size: {len(X_test)}")
print(f"\nClass distribution in training set:")
print(f"  Class 0: {(y_train == 0).sum()}")
print(f"  Class 1: {(y_train == 1).sum()}")

# Calculate scale_pos_weight for class imbalance
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print(f"\nScale pos weight: {scale_pos_weight:.4f}")

Training set size: 269987
Validation set size: 89996
Test set size: 89996

Class distribution in training set:
  Class 0: 219028
  Class 1: 50959

Scale pos weight: 4.2981


# Train model

In [5]:
model = xgb.XGBClassifier(
    learning_rate=LR,
    max_depth=MAX_DEPTH,
    n_estimators=NUM_EPOCHS,
    reg_lambda=L2_REG,  # L2 regularization
    eval_metric='error',  # Classification error rate (1 - accuracy)
    early_stopping_rounds=PATIENCE,
    enable_categorical=False,  # Categorical features are already encoded
    scale_pos_weight=scale_pos_weight,  # Handle class imbalance
    random_state=42,
    min_child_weight = MIN_CHILD_WEIGHT, # Minimum sum of instance weight needed in a child
    colsample_bytree = COSAMPLE_RATE, # Subsample ratio of columns when constructing each tree
    subsample = DATA_RATE # Subsample ratio of the training instances
)

print("TRAINING MODEL")

# Train with evaluation set
model.fit(
    X_train, 
    y_train,
    eval_set=[(X_train, y_train), (X_val, y_val)],
    verbose=True
)

TRAINING MODEL
[0]	validation_0-error:0.27701	validation_1-error:0.31959
[1]	validation_0-error:0.25978	validation_1-error:0.30392
[2]	validation_0-error:0.23852	validation_1-error:0.28779
[3]	validation_0-error:0.23012	validation_1-error:0.28089
[4]	validation_0-error:0.22426	validation_1-error:0.27903
[5]	validation_0-error:0.22127	validation_1-error:0.27871
[6]	validation_0-error:0.21937	validation_1-error:0.27748
[7]	validation_0-error:0.21691	validation_1-error:0.27545
[8]	validation_0-error:0.21611	validation_1-error:0.27537
[9]	validation_0-error:0.21634	validation_1-error:0.27401
[10]	validation_0-error:0.21687	validation_1-error:0.27379
[11]	validation_0-error:0.21637	validation_1-error:0.27362
[12]	validation_0-error:0.21559	validation_1-error:0.27336
[13]	validation_0-error:0.21391	validation_1-error:0.27221
[14]	validation_0-error:0.21433	validation_1-error:0.27302
[15]	validation_0-error:0.21378	validation_1-error:0.27306
[16]	validation_0-error:0.21410	validation_1-error:

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=20,
              enable_categorical=False, eval_metric='error', feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.001, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=20,
              max_leaves=None, min_child_weight=5, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=1000,
              n_jobs=None, num_parallel_tree=None, ...)

# Training results

In [6]:
# Get best iteration
print(f"\nBest iteration: {model.best_iteration}")
print(f"Best validation error: {model.best_score:.4f}")

# Predictions on validation set
y_val_pred = model.predict(X_val)
val_accuracy = accuracy_score(y_val, y_val_pred)
print(f"\nValidation Accuracy: {val_accuracy:.4f}")

# Predictions on test set
y_test_pred = model.predict(X_test)
test_accuracy = accuracy_score(y_test, y_test_pred)
print(f"Test Accuracy: {test_accuracy:.4f}")


Best iteration: 143
Best validation error: 0.2683

Validation Accuracy: 0.7317
Test Accuracy: 0.7286


# Detailed evaluation on test set

In [7]:
print("Classification Report (Test Set):")
print(classification_report(y_test, y_test_pred, target_names=['No Diabetes', 'Diabetes']))

Classification Report (Test Set):
              precision    recall  f1-score   support

 No Diabetes       0.91      0.74      0.82     73010
    Diabetes       0.38      0.69      0.49     16986

    accuracy                           0.73     89996
   macro avg       0.64      0.71      0.65     89996
weighted avg       0.81      0.73      0.75     89996



In [8]:
print("Confusion Matrix (Test Set):")
cm = confusion_matrix(y_test, y_test_pred)
print(cm)

Confusion Matrix (Test Set):
[[53910 19100]
 [ 5323 11663]]


# Save the model

In [9]:
# Save the trained model
dump(model, '../models/xgboost_model.bin')
print("\nModel saved to '../models/xgboost_model.bin'")


Model saved to '../models/xgboost_model.bin'
